In [ ]:
import pandas as pd

# Load IEA wide format data

df_iea = pd.read_csv('../../data/processed/iea_wide_format.csv')

# Load merged global stations dataset

df_stations = pd.read_csv('../../data/processed/merged_charging_station/ev_stations_merged_global.csv')

In [8]:
# Aggregate station count per country (fit new schema)
station_stats = df_stations.groupby('country').agg({
    'id': 'count',
    'operator': lambda x: x.nunique(),
    'status': lambda x: (x == 'Operational').mean(),
    'num_connectors': 'mean'
}).rename(columns={
    'id': 'total_stations',
    'operator': 'unique_operators',
    'status': 'operational_ratio',
    'num_connectors': 'avg_connectors'
})

station_stats.head()

,total_stations,unique_operators,operational_ratio,avg_connectors
country,,,,
AE,3,2,1.000000,1.000000
AL,1,1,1.000000,1.000000
AM,2,1,1.000000,1.000000
AT,43,8,0.976744,1.558140
AU,39,11,1.000000,1.692308


In [9]:
df_merged = df_iea.merge(
    station_stats,
    left_on='region',
    right_index=True,
    how='left'
)

df_merged.head()

,region,year,category,mode,powertrain,ev_charging_points,ev_sales,ev_sales_share,ev_stock,ev_stock_share,electricity_demand,oil_displacement_mbd,"oil_displacement,_million_lge",total_stations,unique_operators,operational_ratio,avg_connectors
0,Australia,2011,Historical,Cars,BEV,NaN,49.0,NaN,49.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Australia,2011,Historical,Cars,EV,NaN,NaN,0.0065,NaN,0.00039,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Australia,2012,Historical,Cars,BEV,NaN,170.0,NaN,220.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Australia,2012,Historical,Cars,EV,NaN,NaN,0.0300,NaN,0.00240,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Australia,2012,Historical,Cars,PHEV,NaN,80.0,NaN,80.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
df_merged['stations_per_million_evs'] = (
    df_merged['total_stations'] / (df_merged['ev_stock'] / 1_000_000)
)

In [11]:
df_merged = df_merged[df_merged['category'] == 'Historical'].copy()

df_merged.head()

,region,year,category,mode,powertrain,ev_charging_points,ev_sales,ev_sales_share,ev_stock,ev_stock_share,electricity_demand,oil_displacement_mbd,"oil_displacement,_million_lge",total_stations,unique_operators,operational_ratio,avg_connectors,stations_per_million_evs
0,Australia,2011,Historical,Cars,BEV,NaN,49.0,NaN,49.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Australia,2011,Historical,Cars,EV,NaN,NaN,0.0065,NaN,0.00039,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Australia,2012,Historical,Cars,BEV,NaN,170.0,NaN,220.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Australia,2012,Historical,Cars,EV,NaN,NaN,0.0300,NaN,0.00240,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Australia,2012,Historical,Cars,PHEV,NaN,80.0,NaN,80.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_merged.to_csv('../../data/processed/merged_dataset.csv', index=False)
print(f"Saved: {len(df_merged)} rows, {len(df_merged.columns)} columns")

Saved: 5085 rows, 18 columns
